# Intro to MicroFVS
MicroFVS is designed to provide access to forest growth-and-yield capabilities using the Forest Vegetation Simulator through a REST API. This notebook provides a gentle, but relatively thorough, introduction. 

We will step through three fundamental components of using MicroFVS in this notebook:
1. Forest inventory input data
2. Keyfiles to define FVS simulation instructions
3. Submitting and receiving requests from MicroFVS web service

In [ ]:
import json
import re

import pandas as pd
import requests
from pydantic import ValidationError

from microfvs.enums import (
    FvsEventType,
    FvsKeyfileTemplate,
    FvsOutputTableName,
    FvsVariant,
)
from microfvs.models import (
    FvsEventLibrary,
    FvsKeyfile,
    FvsResult,
    FvsStandInit,
    FvsTreeInit,
    FvsTreeInitRecord,
)

## 1. Forest Inventory Inputs

Here we will cover the core data formats that will allow us to create or load forest inventory data for use in simulations, and to ensure that it passes some basic quality assurance checks. MicroFVS currently provides for the execution of a single simulation for a single stand as the minimal starting point. This starting point is intended to be extended to support batch modeling capabilities soon.

Our primary building blocks for forest inventory data are: `FvsStandInit`, `FvsTreeInitRecord`, and `FvsTreeInit`.

`FvsStandInit` contains the attributes of a single stand.

In [ ]:
FvsStandInit?

Let's make a new `StandInitRecord` with the minimum required fields ...

In [ ]:
stand_init = FvsStandInit(
    stand_id="12345",
    variant="PN",
    inv_year=2020,
    basal_area_factor=-10,  # 1/10th-acre fixed radius plot for large trees
    inv_plot_size=100,  # 1/100th-acre fixed radius plot for small trees
    brk_dbh=5,  # large trees start at 5" DBH
)
stand_init

At the tree-level, our equivalent minimal data container is an `FvsTreeInitRecord`. This stores the attributes of a single tree.

In [ ]:
FvsTreeInitRecord?

We combine multiple `FvsTreeInitRecord`s into an `FvsTreeInit` object, which represents the trees present within a single stand we intend to use for simulation.

In [ ]:
tree1 = FvsTreeInitRecord(
    stand_id="12345", plot_id=1, tree_id=1, species="DF", diameter=10.5
)
tree2 = FvsTreeInitRecord(
    stand_id="12345", plot_id=1, tree_id=2, species=242, dbh=12.0
)
seed1 = FvsTreeInitRecord(
    stand_id="12345", plot_id=1, tree_id=3, species=263, dbh=0.1, tree_count=2
)
tree_init = FvsTreeInit(trees=[tree1, tree2, seed1])
tree_init

These `FvsTreeInitRecord`, `FvsTreeInit` and `FvsStandInit` classes are implemented as [Pydantic BaseModels](https://docs.pydantic.dev/latest/), so they come along with type-checking and other validation settings that will tell you when you've done something that really won't work. 

These checks are intended to be helpful and catch more egregious data entry errors, but are not exhaustive. For example, we don't check for species codes FVS doesn't recognize, that your trees aren't wider than they are tall, or that the field crew might have been using the wrong side of the D-tape!

In [ ]:
try:  # missing several required fields
    FvsStandInit(stand_id="12345")
except ValidationError as e:
    print(e)

In [ ]:
try:  # negative DBH is invalid
    FvsTreeInitRecord(stand_id="123", species=202, dbh=-1.0)
except ValidationError as e:
    print(e)

You can go back and forth between `FvsStandInit` and `FvsTreeInit` objects and `DataFrame`s using simple helper methods `from_dataframe` and `to_dataframe`. Because MicroFVS is intended to process simulations submitted one stand at a time, each time you create an instance of a `FvsStandInit` or `FvsTreeInit` object from a dataframe, you need to specify the `stand_id` that should be retained. This will leave out any data in the input `DataFrame` for stands with a different `stand_id`.

In [ ]:
# Use your favorite data source to create the dataframe
# such as `tree_init_df = pd.read_csv("trees.csv")`

tree_init_df = pd.DataFrame(
    {
        "stand_id": ["12345", "12345", "6789"],
        "plot_id": [1, 1, 1],
        "tree_id": [1, 2, 3],
        "species": ["DF", "WH", "RC"],
        "diameter": [10.5, 1.0, 0.1],
        "tree_count": [150, 25, 25],
    }
)

# notice that the tree from stand_id "6789" is no longer included
FvsTreeInit.from_dataframe(df=tree_init_df, stand_id="12345").to_dataframe()

## 2. FVS Keyfiles and Templating
Under the hood, MicroFVS executes the Forest Vegetation Simulator as a command-line tool, feeding it a single **FVS Keyfile**. 

A Keyfile is a plain text file containing a set of **FVS Keywords** representing simulation instructions for FVS. Keywords are well-documented in the FVS [Keyword Reference Guide](https://www.fs.usda.gov/sites/default/files/fvs-keyword.pdf), in the [Fire and Fuels Extension](https://www.fs.usda.gov/sites/default/files/forest-management/fvs-ffe-guide.pdf) and in the [Users Guide to the Database Extension](https://www.fs.usda.gov/sites/default/files/forest-management/fvs-dbs-user-guide.pdf). A set of FVS Keywords is commonly referred to as Keyword Components, and are often saved as `*.kcp` files.

We consider an individual FVS simulation primarily defined by the regional variant used for the simulation, the specific stand being simulated, and the sequences of treatments (if any) and natural disturbances (if any) that are called for. 

In MicroFVS, we generate keyfiles using a `FvsKeyfile` class, which accepts each of these arguments, along with a `template` and set of `template_params` that can be injected. We will wade into more detail how to provide `treatments` and `disturbances` further below.

In [ ]:
FvsKeyfile?

We provide a default FVS keyfile template. This is a long multi-line string that includes placeholders with double curly braces: `{{placeholder_name}}`.  These placeholders are used to inject values when the keyfile template is rendered. When the template is rendered, each parameter gets injected into the placeholder that bears its name. 

This keyfile template is implemented using the [Jinja2 templating system](https://jinja.palletsprojects.com/en/stable/templates/). Jinja2 provides a rich set of options for constructing templates beyond the handful we've employed in this default keyfile template. We encourage you to check out the Jinja2 documentation if you feel drawn to make your own FVS keyfile template for uses beyond those ours will accomodate.

In [ ]:
print(FvsKeyfileTemplate.DEFAULT)

The default template provides several placeholders where you can inject arbitrary keyword components to be included for the FVS simulation. Don't worry if you don't have something to fill all the placeholders, most of these employ sensible defaults (e.g., `cycle_length == 5`, `num_cycles == 1`). 

In [ ]:
print(FvsKeyfileTemplate.DEFAULT.get_template_params())

We feed the parameters we want to inject into the template using a `FvsKeyfile` object. The only parameters you must always provide are the `variant` and the `stand_id`. If you don't specify any treatments or disturbances, a "Grow Only" scenario will be generated.

In [ ]:
my_simple_keyfile = FvsKeyfile(
    variant=FvsVariant.PN,
    stand_id=12345,
    template_params={"cycle_length": 5, "num_cycles": 10},
)
print(my_simple_keyfile)

Apart from basic configuration settings like the cycle length and number of simulation cycles you want to run, the next most commonly used params will probably be the specification of treatments and disturbance events. We provide a library of treatments and disturbances that you can invoke by name. We have scraped all the examples from a [national compendium of silvicultural treatments](https://doi.org/10.2737/RDS-2022-0037) prepared by a team of silviculture experts from the National Forest System and Research Station of the US Forest Service in the form of FVS Keyword Component (KCP) files, and you can access them by short-hand codes and inject them directly into a template.

You can access these USFS treatments from the library. 

In [ ]:
library = FvsEventLibrary()
# library.usfs_treatments is a dictionary with the treatment name as the
# key and an FvsEvent object as the value
print(sorted(library.usfs_treatments.keys()))

We'll now create a few events to include in our next simulation. 

We'll start with treatment `06-03`, a commercial thinning prescription developed in USFS Region 6. 

We included a `lookup` helper method allow events to easily inspect items in the library.

In [ ]:
COMMERCIAL_THINNING_KEY = "06-03"

thinning_treatment = library.lookup(
    event_type=FvsEventType.TREATMENT,
    event_key=COMMERCIAL_THINNING_KEY,
)
print(thinning_treatment)

We'll utilize a disturbance available in the library as well.

In [ ]:
INTENSE_FIRE_KEY = "FIC6"
intense_fire = library.lookup(
    event_type=FvsEventType.DISTURBANCE, event_key=INTENSE_FIRE_KEY
)
print(intense_fire)

We can inject the extended set of params into the `FvsKeyfile`. 

We create a quick helper function here to zoom to the specific segments of the rendered keyfile so you won't have to scroll through the whole thing.

In [ ]:
def zoom_to_pattern(
    text: str, pattern: str, lines_after: int = 15, lines_before: int = 0
) -> str:
    """Zooms to a line matching a pattern and shows what follows it.

    Args:
        text (str): The text body to search.
        pattern (str): The pattern to search for in ``text``.
        lines_after (int): Number of lines to show after the match.
        lines_before (int): Number of lines to show before the match.
    """
    lines = text.splitlines()

    for i, line in enumerate(lines):
        if re.search(pattern, line):
            safe_start = max(0, i - lines_before)
            safe_end = min(len(lines), i + lines_after + 1)
            result = "\n".join(lines[safe_start:safe_end])
            if safe_start > 0:
                result = "...\n" + result
            if safe_end < len(lines):
                result = result + "\n..."
            return result

    return "Pattern not found."

In [ ]:
extended_keyfile = FvsKeyfile(
    variant="PN",
    stand_id="12345",
    treatments=[thinning_treatment],
    disturbances=[intense_fire],
    template_params={"cycle_length": 5, "num_cycles": 10},
)

In [ ]:
treatment_lines_after = len(thinning_treatment.content.splitlines())
print(
    zoom_to_pattern(
        extended_keyfile.content,
        r"MANAGEMENT TREATMENTS",
        lines_after=treatment_lines_after + 1,
        lines_before=1,
    )
)

In [ ]:
disturbance_lines_after = len(intense_fire.content.splitlines())
print(
    zoom_to_pattern(
        extended_keyfile.content,
        r"NATURAL DISTURBANCES",
        lines_after=disturbance_lines_after + 2,
        lines_before=1,
    )
)

### Roll Your Own
You are welcome to create your own keyfile templates. We demonstrate that here for thoroughness, although we'd strongly encourage you to try and work with the default template first, which we've designed to be very flexible. 

Making your own template is relatively simple to get started, but if you're not careful to follow the spacing requirements, you might introduce unexpected bugs into the simulation. And FVS may or may not give you useful error messages.

In the example below, there are no commands to read in stand or tree data, no FVS simulation settings are configured, and there is no call to FVS to process the keyfile. This is not a runnable keyfile, but it can demonstrate the templating process, including the use of nested placeholders within keyword components.

In [ ]:
CUSTOM_KEYFILE_TEMPLATE = """
** {{ helpful_comment }}
{{ some_keywords }}
{{ keywords_with_nested_placeholder }}
** {{ unhelpful_comment }}
"""

STATIC_KEYWORD_COMPONENTS = """
** This demonstrates injection of static keyword components.
COMPUTE            0
DF_TPA = SPMCDBH(1,DF,0)
END
"""

KEYWORDS_WITH_NESTED_PLACEHOLDER = """
** This demonstrates the use of a nested placeholder.
COMPUTE            0
{{ nested_placeholder_1 }} = SPMCDBH(1,{{ nested_placeholder_2 }},0)
END
"""

custom_keyfile = FvsKeyfile(
    variant="PN",
    stand_id="87654",
    template=CUSTOM_KEYFILE_TEMPLATE,
    template_params={
        "helpful_comment": "Custom Keyfile version 0.1.0",
        "some_keywords": STATIC_KEYWORD_COMPONENTS,
        "keywords_with_nested_placeholder": KEYWORDS_WITH_NESTED_PLACEHOLDER,
        "unhelpful_comment": "I wonder what this button does.",
        "nested_placeholder_1": "RC_TPA",
        "nested_placeholder_2": "RC",
    },
)

print(custom_keyfile)

Instead of rolling your own keyfile template, we'd generally encourage you to generate keyword components that can be injected into the default template. 

For example, if you wanted to enter a custom SDIMAX keyword, you could do it like this.

In [ ]:
MY_SDIMAX = "SDIMAX           All       600"

You might want to be very careful making raw strings though, because FVS is particular about the spacing of keywords and their arguments. It's safer to enforce spacing and column alignment of keyword arguments using an approach like this.

In [ ]:
def make_fixed_width_keyword(
    keyword,
    f1=None,
    f2=None,
    f3=None,
    f4=None,
    f5=None,
    f6=None,
    f7=None,
):
    """A simplistic function to make fixed-width keywords."""
    fs = {}
    for i, f in enumerate([f1, f2, f3, f4, f5, f6, f7]):
        fs[i] = f or ""
    return (
        f"{keyword:<10}{fs[0]:>10}{fs[1]:>10}{fs[2]:>10}{fs[3]:>10}"
        f"{fs[4]:>10}{fs[5]:>10}{fs[6]:>10}"
    )


df_sdimax = make_fixed_width_keyword(keyword="SDIMAX", f1="DF", f2=600)
wh_sdimax = make_fixed_width_keyword(keyword="SDIMAX", f1="WH", f2=1000)
my_sdimax_kcp = "\n".join([df_sdimax, wh_sdimax])

We can see the custom keyword component injected at the `sdimax` placeholder we have in the default keyfile template.

In [ ]:
custom_sdimax_keyfile = FvsKeyfile(
    variant="PN",
    stand_id="87654",
    template_params={"sdimax": my_sdimax_kcp},
)

print(
    zoom_to_pattern(
        custom_sdimax_keyfile.content,
        r"SDIMAX SETTINGS",
        lines_after=len(my_sdimax_kcp.splitlines()) + 1,
        lines_before=1,
    )
)

## 3. Interacting with the MicroFVS REST API
Now we're ready to see how we can interact with MicroFVS as a web service using `GET` and `POST` requests. As we develop client libraries for interacting with MicroFVS through Python and R, we expect to simplify these use cases substantially so that users are not expected to specify `GET` and `POST` requests themselves, nor to parse results back into formats like DataFrames. 

We will demonstrate a few key endpoints of the MicroFVS REST API to which requests can be submitted:
* **Version Check:** Which versions are being run for each FVS variant in the web service?
* **Run FVS Simulation:** Execute FVS with your own inventory data, using the default keyfile template and grow-only scenario.
* **Run FVS with Treatment & Disturbance:** Insert a thinning treatment before an intense wildfire.


In [ ]:
MICROFVS_BASE_URL = "http://localhost:8000"
TEMPLATE_ENDPOINT = MICROFVS_BASE_URL + "/template"
VERSION_ENDPOINT = MICROFVS_BASE_URL + "/version"
KEYFILE_ENDPOINT = MICROFVS_BASE_URL + "/keyfile"
RUN_ENDPOINT = MICROFVS_BASE_URL + "/run"

### Check the FVS Versions
In this first instance, we use a `GET` request to the `/versions` endpoint. The result is a dictionary.

In [ ]:
response = requests.get(VERSION_ENDPOINT)
pretty_json = json.dumps(response.json(), indent=4)
print(pretty_json)

### Executing FVS (grow only)
Here we demonstrate a couple simulations triggered using a `POST` request to the `/run` endpoint. 

We construct some simple stand and tree initialization data here, but you could also utilize the `FvsStandInit.from_dataframe` and `FvsTreeInit.from_dataframe` approaches with any datasets you can load into dataframe format. 

In [ ]:
stand_init = FvsStandInit(
    stand_id="12345",
    variant="PN",
    inv_year=2020,
    basal_area_factor=-10,  # 1/10th-acre fixed radius plot for large trees
    inv_plot_size=100,  # 1/100th-acre fixed radius plot for small trees
    brk_dbh=5,  # large trees start at 5" DBH
)

tree1 = FvsTreeInitRecord(
    stand_id="12345",
    plot_id=1,
    tree_id=1,
    species="DF",
    diameter=14.5,
    tree_count=6,
)
tree2 = FvsTreeInitRecord(
    stand_id="12345",
    plot_id=1,
    tree_id=2,
    species="WH",
    dbh=12.0,
    tree_count=3,
)
seed1 = FvsTreeInitRecord(
    stand_id="12345", plot_id=1, tree_id=3, species="RC", dbh=0.1, tree_count=2
)
tree_init = FvsTreeInit(trees=[tree1, tree2, seed1])

The response from the `/run` endpoint is an instance of `FvsResult` transmitted back to us in JSON format. `FvsResult` combines metadata about the simulation, as well as the output data. 

We have scraped some fundamental metadata from these simulations beyond what FVS itself usually provides in output forms to help clearly document the computation and to make it more reproducible and auditable: 
* **`command`:** the command-line call that was issued to trigger the FVS simulation.
* **`return_code`:** the return code from the command line which indicates the severity of problems (if any) that FVS encountered during the simulation.
* **`stdout`:** the Standard Output echoed to the terminal during the FVS simulation.
* **`stderr`:** the Standard Error echoed to the terminal during the FVS simulation, which may indicate when FVS experiences a catastrophic failure (e.g., a segmentation fault), in which case the resulting outputs may be fully or partially incomplete.
* **`fvs_warnings`:** FVS warnings and errors that were detected in the plain text output file.

In [ ]:
basic_run_response = requests.post(
    RUN_ENDPOINT,
    json={
        "stand_init": stand_init.model_dump(),
        "tree_init": tree_init.model_dump(),
        "template_params": {
            "num_cycles": 10,
            "cycle_length": 5,
            # ... other parameters you want to inject into the template
        },
    },
)

# parse the response from JSON back into an FvsResult object
basic_run_result = FvsResult.model_validate(basic_run_response.json())
print(basic_run_result)

This `print(basic_run_result)` above just shows a print-friendly summary of the `FvsResult`. The two other attributes you might want to inspect are the content of the plain-text keyfile that was used to execute the simulation...

In [ ]:
print(basic_run_result.keyfile)

... as well as the plain-text outfile that FVS generated. This output may be particularly useful for debugging problematic simulations, since it will often indicate where in the sequence of simulation instructions an error or warning was encountered.

In [ ]:
print(basic_run_result.outfile)

While some of us FVS nerds enjoy nothing more than curling up in a comfy chair beside the fireplace to review plain-text FVS input and output files, most well-adjusted users of this API will no doubt be more interested in the tabular output data generated by FVS. 

The `FvsResult.fvs_data` is the gateway to all of the data tables we scraped from the FVS database which was created on the web server to hold both the FVS inventory data inputs and the simulation outputs. The data tables are stored in a dictionary, and you can access them by name (e.g., `result.fvs_data["fvs_summary2"]`). To avoid potential issues around case-sensitive string lookups, we generally rely upon enumerations of accepted string values like the `FvsOutputTableName` class, which is an enumeration of accepted FVS output table names. 

In [ ]:
basic_run_result.fvs_data[FvsOutputTableName.FVS_SUMMARY2]

### A More Eventful FVS Simulation
Here we will employ a couple events from the MicroFVS `FvsEventLibrary` demonstrated above: the commercial thinning treatment defined by the USFS; and the the intense wildfire event defined by Vibrant Planet.

In [ ]:
eventful_run_response = requests.post(
    RUN_ENDPOINT,
    json={
        "stand_init": stand_init.model_dump(),
        "tree_init": tree_init.model_dump(),
        "template_params": {"num_cycles": 10, "cycle_length": 5},
        # we submit the treatment and disturbance event keys as strings
        # to let the web service look them up from the library for us
        "treatments": [COMMERCIAL_THINNING_KEY],
        "disturbances": [INTENSE_FIRE_KEY],
    },
)
# parse the response from JSON back into an FvsResult object
eventful_run_result = FvsResult.model_validate(eventful_run_response.json())
print(eventful_run_result)

You can inspect the `FvsResult.keyfile` attribute if you want to confirm that the additional instructions got inserted. You can also just look in the FVS output tables and see that harvest removals did occur in 2020 and a fire was triggered in 2035.

In [ ]:
# print(eventful_run_result.keyfile)
eventful_run_result.fvs_data[FvsOutputTableName.FVS_SUMMARY2]

In [ ]:
eventful_run_result.fvs_data[FvsOutputTableName.FVS_CONSUMPTION]

In [ ]:
eventful_run_result.fvs_data[FvsOutputTableName.FVS_BURN_REPORT]

### Passing a custom set of treatments and/or disturbance events.

If we want to pass on or more custom treatments or disturbances, we need to provide them with a name and with the keyword components as content.

In [ ]:
thin_sdi_2030_kcp = make_fixed_width_keyword(
    keyword="THINSDI", f1="2030", f2="200"
)
thin_sdi_2050_kcp = make_fixed_width_keyword(
    keyword="THINSDI", f1="2050", f2="250"
)

custom_run_response = requests.post(
    RUN_ENDPOINT,
    json={
        "stand_init": stand_init.model_dump(),
        "tree_init": tree_init.model_dump(),
        "template_params": {"num_cycles": 10, "cycle_length": 5},
        # each custom event that can't be looked up in the
        # FvsEventLibrary by name needs to be submitted with its own
        # name and content
        "treatments": [
            {"name": "thin-sdi-2030", "content": thin_sdi_2030_kcp},
            {"name": "thin-sdi-2050", "content": thin_sdi_2050_kcp},
        ],
    },
)
custom_run_result = FvsResult.model_validate(custom_run_response.json())
print(custom_run_result)

We can see that the thins were triggered when we called for them, and that the after-thinning SDI matches the targets we specified (check out the rows with `rmvcode == 2` to see after-treatment values and `rmvcode == 1` for the before-treatment values).

In [ ]:
DISPLAY_COLS = [
    "standid",
    "year",
    "rmvcode",
    "tpa",
    "ba",
    "qmd",
    "sdi",
    "bdft",
    "rbdft",
]
custom_run_result.fvs_data[FvsOutputTableName.FVS_SUMMARY2][DISPLAY_COLS]